# Mutual Information-based Registration

## Introduction
In this exercise, you will implement 3D rigid image registration using mutual information (MI). You will work with a T1-weighted and a T2-weighted MRI scan of the same patient, where the patient is lying in a different position in the scanner between the two image acquisition (for the purpose of this exercise, the movement is simulated).

### Instructions for your report
Your report should be structured as follows:
* **Introduction**: a short summary of what your report is about, perhaps with a recap of some of the equations you'll be using in your solution.
* **Task 1, ..., 5**: for each specific task listed below, your code (in code cells), your results (as figures) as well as explanations of what is computed and what is shown in the figures (in Markdown cells). 
* **Conclusion**: a short summary of your findings.

This introduction should **not** be part of your report.

### Input data and code hints
Import Python libraries:

In [1]:
import numpy as np
from numpy import linalg as la
import matplotlib.pyplot as plt
import matplotlib as mpl
from mpl_toolkits import mplot3d
#%matplotlib notebook
%matplotlib tk
plt.ion()
np.set_printoptions( suppress=True )
import nibabel as nib
import scipy
from scipy.ndimage import map_coordinates

Read the two 3D scans you'll be working with in this exercise:

In [2]:
T1_fileName = 'IXI014-HH-1236-T1.nii.gz'
T2_fileName = 'IXI014-HH-1236-T2_rotated.nii.gz'
T1 = nib.load( T1_fileName )
T2 = nib.load( T2_fileName )
T1_data = T1.get_fdata()
T2_data = T2.get_fdata()

Below is code to define a simple interactive viewer class that can be used to visualize 2D cross-sections of a 3D array along three orthogonal directions. It takes a 3D volume as input and shows the location a "linked cursor" in all three cross-sections.

The initial location of the cursor is in the middle of the volume in each case. It can be changed by clicking on one of the cross-sections. The viewer also displays the voxel index $\mathbf{v}$ of the cursor.

In [3]:
class Viewer:
    def __init__(self, data ):
        self.fig, self.ax = plt.subplots()
        self.data = data
        self.dims = self.data.shape
        self.position = np.round( np.array( self.dims ) / 2 ).astype( int )
        self.draw()
        self.fig.canvas.mpl_connect( 'button_press_event', self )
        self.fig.show()

    def __call__(self, event):
        print( 'button pressed' )
        if event.inaxes is None: return
      
        x, y = round( event.xdata ), round( event.ydata )

        #
        if ( x > (self.dims[0]-1) ) and ( y <= (self.dims[1]-1) ): return # lower-right quadrant
          
        #
        if x < self.dims[0]:
          self.position[ 0 ] = x
        else:
          self.position[ 1 ] = x - self.dims[0]
        
        if y < self.dims[1]:
          self.position[ 1 ] = y
        else:
          self.position[ 2 ] = y -self.dims[1]
        
        print( f"  voxel index: {self.position}" )
        print( f"  intensity: {self.data[ self.position[0], self.position[1], self.position[2] ]}" )

        self.draw()

    def draw( self ):
        #
        # Layout on screen is like this:
        #
        #     ^            ^
        #  Z  |         Z  |
        #     |            |
        #     ----->        ---->  
        #       X             Y
        #     ^
        #  Y  |
        #     |
        #     ----->  
        #       X
        #
        dims = self.dims
        position = self.position
        
        xySlice = self.data[ :, :, position[ 2 ] ]
        xzSlice = self.data[ :, position[ 1 ], : ]
        yzSlice = self.data[ position[ 0 ], :, : ]
        
        kwargs = dict( vmin=self.data.min(), vmax=self.data.max(), 
                       origin='lower', 
                       cmap='gray',
                       picker=True )

        self.ax.clear()

        self.ax.imshow( xySlice.T, 
                        extent=( 0, dims[0]-1, 
                                 0, dims[1]-1 ), 
                        **kwargs )
        self.ax.imshow( xzSlice.T, 
                        extent=( 0, dims[0]-1, 
                                 dims[1], dims[1]+dims[2]-1 ), 
                        **kwargs )
        self.ax.imshow( yzSlice.T, extent=( dims[0], dims[0]+dims[1]-1, 
                                            dims[1], dims[1]+dims[2]-1 ), 
                        **kwargs )

        color = 'g'
        self.ax.plot( (0, dims[0]-1), (position[1], position[1]), color )
        self.ax.plot( (0, dims[0]+dims[1]-1), (dims[1]+position[2], dims[1]+position[2]), color )
        self.ax.plot( (position[0], position[0]), (0, dims[1]+dims[2]-1), color )
        self.ax.plot( (dims[0]+position[1], dims[0]+position[1]), (dims[1]+1, dims[1]+dims[2]-1), color )

        self.ax.set( xlim=(1, dims[0]+dims[1]), ylim=(0, dims[1]+dims[2]) )

        self.ax.text( dims[0] + dims[1]/2, dims[1]/2, 
                      f"voxel index: {position}",  
                      horizontalalignment='center', verticalalignment='center' )
  
        self.ax.axis( False )

        self.fig.canvas.draw()

### Task 1: Resample the T2-weighted scan to the image grid of the T1-weighted scan

In this task you perform the resampling just like in the landmark-based registration exercise. Find the voxel-to-voxel transformation between $\mathbf{v}_{T1}$ and $\mathbf{v}_{T2}$:

$$
\begin{pmatrix} \mathbf{ v_{T2}} \\ 1 \end{pmatrix} = \mathbf{M}_{T2}^{-1} \cdot \mathbf{M}_{T1} \cdot \begin{pmatrix} \mathbf{ v_{T1}} \\ 1 \end{pmatrix}
.
$$
At the location $\mathbf{v}_{T2}$, you should then use cubic B-spline interpolation to determine the intensity in the T2-weighted scan, and store it at index $\mathbf{v}_{T1}$ in the newly created image.

> ***Hints:***
> - you can create a coordinate grid in 3D with the function
> 
>        V1,V2,V3 = np.meshgrid( np.arange( T1_data.shape[0] ), 
>                                np.arange( T1_data.shape[1] ), 
>                                np.arange( T1_data.shape[2] ), indexing='ij' )
>   
>
> - the following SciPy function interpolates the T2-weighted volume at voxel coordinates $(1.1,2.2,3.3)^T$ 
> and $(6.6,7.7,8.8)^T$ using cubic interpolation:   
>
>        scipy.ndimage.map_coordinates( T2_data, np.array( [ [1.1,2.2,3.3], [6.6,7.7,8.8] ] ).T )

In [4]:
V1,V2,V3 = np.meshgrid( np.arange( T1_data.shape[0] ),
                        np.arange( T1_data.shape[1] ), 
                        np.arange( T1_data.shape[2] ), indexing='ij' )


In [5]:
v_T1 = np.vstack([V1.ravel(), V2.ravel(), V3.ravel(), np.ones(V1.size)])
print(v_T1)   

[[  0.   0.   0. ... 255. 255. 255.]
 [  0.   0.   0. ... 255. 255. 255.]
 [  0.   1.   2. ... 147. 148. 149.]
 [  1.   1.   1. ...   1.   1.   1.]]


In [6]:
M_T1 = T1.affine          
M_T2 = T2.affine          
M_T1_to_T2 = la.inv(M_T2) @ M_T1   

v_T2 = M_T1_to_T2 @ v_T1
print(v_T2.shape)

(4, 9830400)


In [7]:
print(T1.affine)

[[   0.01506851   -0.01498772    1.19969177  -89.9808197 ]
 [  -0.92914075    0.12376414    0.02165389  117.73051453]
 [   0.12400278    0.92917383    0.01646696 -121.56955719]
 [   0.            0.            0.            1.        ]]


Once you have created the resampled T2-weighted data, visualize the T1-weighted and resampled T2-weighted volumes overlaid as follows:
    
        Viewer( T2_data_resampled / T2_data_resampled.max() + T1_data / T1_data.max() )

 Can you determine around what axis (in world coordinates) the rotation was simulated?

 > ***Hint***
 > - The T2-weighted volume was rotated around a single axis only.
 >

In [8]:
points = v_T2[:3, :].T                       # (N, 3): one row per point, like the hint's array
T2_data_resampled = scipy.ndimage.map_coordinates(T2_data, points.T)   # .T gives (3, N), as in the hint
T2_data_resampled = T2_data_resampled.reshape(T1_data.shape)

print(T1_data.shape, T2_data.shape, T2_data_resampled.shape)

(256, 256, 150) (256, 256, 28) (256, 256, 150)


In [9]:
Viewer(T2_data_resampled / T2_data_resampled.max() + T1_data / T1_data.max())

### Task 2: Compute and visualize the joint histogram

Compute the joint histogram 
$$ 
\mathbf{H} = 
\begin{pmatrix}
h_{1,1} & ... & h_{1,B} \\
\vdots & ... & \vdots \\
h_{B,1} & ... & h_{B,B} \\
\end{pmatrix} 
$$
of the T1-weighted scan and the resampled T2-weighted scan you just created, using $B=32$ bins. In order not to let the registration process be dominated by the (extremely numerous) background voxels, ignore all voxels with an intensity lower than 10 in either image when computing the histogram. Visualize the joint histogram using a 3D bar plot, and label the axes.

> ***Hints:***
>
> - To compute a 2D histogram, you can use the function
>
>           np.histogram2d(x,y,bins,range)
> 
> - A 3D bar plot can be created as follows:
>
>           fig, ax = plt.subplots( subplot_kw=dict( projection='3d') )
>           plt.bar3d(xpos, ypos, zpos, dx, dy, dz)
>
>   where xpos, ypos, and zpos are the anchor points of the bars and dx, dy, and dz are the dimensions of the bars.
>
> - A useful "trick" to verify that your axes are labeled correctly, is to use a different number of bins for the two scans.
>

In [10]:
def joint_histogram(F, M, bins=32, threshold=10,
                    range_F=None, range_M=None):
    
    # 1. keep only voxels that are foreground in BOTH images
    mask = (F >= threshold) & (M >= threshold)
    f_vals = F[mask]          # 1D array of T1 intensities
    m_vals = M[mask]          # 1D array of the matching T2 intensities

    # 2. fixed intensity ranges, so the bins are the same every time we call this
    if range_F is None:
        range_F = [threshold, F.max()]
    if range_M is None:
        range_M = [threshold, M.max()]

    # 3. count the pairs
    H, f_edges, m_edges = np.histogram2d(f_vals, m_vals, bins=bins,
                                         range=[range_F, range_M])
    return H, f_edges, m_edges

range_T1 = [10, T1_data.max()]
range_T2 = [10, T2_data.max()]

H, f_edges, m_edges = joint_histogram(T1_data, T2_data_resampled, bins=32,
                                      range_F=range_T1, range_M=range_T2)
print(H.shape)          
print(H.sum())          

(32, 32)
2922740.0


In [11]:
fig, ax = plt.subplots(subplot_kw=dict(projection='3d'))

xpos, ypos = np.meshgrid(np.arange(H.shape[0]), np.arange(H.shape[1]), indexing='ij')
xpos = xpos.ravel()
ypos = ypos.ravel()
zpos = np.zeros_like(xpos)      
dz = H.ravel()                  

ax.bar3d(xpos, ypos, zpos, 0.8, 0.8, dz, shade=True)

ax.set_xlabel('T1 intensity bin (f)')
ax.set_ylabel('T2 intensity bin (m)')
ax.set_zlabel('count $h_{f,m}$')
ax.set_title('Joint histogram before registration')
plt.show()



### Task 3: Compute the mutual information between the two images
Write a function that takes two image volumes defined on the same image grid as input, and returns the mutual information between the two images:

$$ 
MI = H_{F} + H_{M} - H_{F,M} 
$$
with
$$ 
H_{F,M} = - \sum_{f=1}^{B} \sum_{m=1}^{B} p_{f,m} \log(p_{f,m}),
$$
$$ 
H_{F} = - \sum_{f=1}^{B} p_{f} \log(p_{f}), 
$$
and
$$
H_{M} = - \sum_{m=1}^{B} p_{m} \log(p_{m})
.
$$

Your function should make use of the joint histogram computed as in the previous task (i.e., using $B=32$ bins and ignoring all voxels with intensity lower than 10).

Use your new function to compute the mutual information between the T1-weighted image and the resampled T2-weighted image.

> ***Hints:***
>
> - To avoid numerical errors when one of the bins is empty and 0*log(0)=0 is therefore computed numerically, you can add a tiny value (e.g., 1e-12) to each histogram bin

In [12]:
def mi_from_histogram(H, eps=1e-12):
    H = H + eps                      
    p_fm = H / H.sum()               
    p_f = p_fm.sum(axis=1)           
    p_m = p_fm.sum(axis=0)           

    H_fm = -np.sum(p_fm * np.log(p_fm))   
    H_f  = -np.sum(p_f  * np.log(p_f))    
    H_m  = -np.sum(p_m  * np.log(p_m))    

    return H_f + H_m - H_fm


def mutual_information(F, M, bins=32, threshold=10, range_F=None, range_M=None):

    H, _, _ = joint_histogram(F, M, bins=bins, threshold=threshold,
                              range_F=range_F, range_M=range_M)
    return mi_from_histogram(H)


MI = mutual_information(T1_data, T2_data_resampled, bins=32,
                        range_F=range_T1, range_M=range_T2)
print(f"MI before registration: {MI:.4f}")

MI before registration: 0.1160


### Task 4: Evaluate the mutual information across a range of rotation angles

Implement a grid search over a range of rotation angles, with the goal of identifying the angle with approximately the highest mutual information (i.e., where the registration is best). Make an educated guess of a suitable range by visually inspecting the images (or by simply trying a few rotation angles and inspecting the result), and then define a list of candidate angles at intervals of e.g., 5° apart:

       a = np.arange(a_{lowest}, a_{highest}, 5)

Loop over all candidate rotation angles, each time (1) creating a corresponding rotation matrix $\mathbf{R}$; (2) resampling the T2-weighted volume accordingly; (3) calculating the mutual information with the function you created above; and (4) storing the mutual information value.

> ***Hint:***
> 
> - A 3D rotation matrix can be parameterized as follows:
> 
> $$\mathbf{R} = \mathbf{R}_{z} \mathbf{R}_{y} \mathbf{R_{x}},$$ 
> where
>
> $$
\mathbf{R}_z = 
\begin{pmatrix} 
 cos( \theta_z ) & -sin( \theta_z ) & 0 & \\
 sin( \theta_z ) & cos( \theta_z ) & 0 & \\
 0 & 0 & 1 & \\
\end{pmatrix}
$$
> implements a rotation around the z-axis in world coordinates;
> $$
\mathbf{R}_y = 
\begin{pmatrix} 
 cos( \theta_y ) & -sin( \theta_y ) & 0 & \\
 0 & 1 & 0 & \\
 sin( \theta_y ) & cos( \theta_y ) & 0 & \\
\end{pmatrix}
$$
> is a rotation around the y-axis;
and
> $$
\mathbf{R}_x = 
\begin{pmatrix} 
 1 & 0 & 0 & \\
 cos( \theta_x ) & -sin( \theta_x ) & 0 & \\
 sin( \theta_x ) & cos( \theta_x ) & 0 & \\
\end{pmatrix}
$$
> rotates around the x-axis.
>
>
> - If you didn't figure out in Task 1 which axis the T2-weighted volume was rotated around, you can try rotating around each axis (one at a time) to see the effect of each transformation. Once you've determined the correct rotation axis, the rotation angles around the other axes should be clamped to zero.


In [14]:
def rotation_matrix(theta_x=0.0, theta_y=0.0, theta_z=0.0):
    """3D rotation R = Rz @ Ry @ Rx, angles in degrees."""
    ax, ay, az = np.deg2rad([theta_x, theta_y, theta_z])
    Rx = np.array([[1, 0, 0],
                   [0, np.cos(ax), -np.sin(ax)],
                   [0, np.sin(ax),  np.cos(ax)]])
    Ry = np.array([[ np.cos(ay), 0, np.sin(ay)],
                   [0, 1, 0],
                   [-np.sin(ay), 0, np.cos(ay)]])
    Rz = np.array([[np.cos(az), -np.sin(az), 0],
                   [np.sin(az),  np.cos(az), 0],
                   [0, 0, 1]])
    return Rz @ Ry @ Rx


def resample_T2(R, order=3):
    """Resample T2 onto the T1 grid after applying rotation R in world coordinates."""
    M_R = np.eye(4)
    M_R[:3, :3] = R
    M = la.inv(T2.affine) @ M_R @ T1.affine
    v_T2 = M @ v_T1
    out = map_coordinates(T2_data, v_T2[:3, :], order=order,
                          mode='constant', cval=0.0)
    return out.reshape(T1_data.shape)


# Sanity check: no rotation must reproduce the Task 1 result
print(np.allclose(resample_T2(rotation_matrix()), T2_data_resampled))


angles = np.arange(-180, 180, 5)

mi_values = []
for a in angles:
    T2_rot = resample_T2(rotation_matrix(theta_z=a), order=1)
    mi_values.append(mutual_information(T1_data, T2_rot, bins=32,
                                        range_F=range_T1, range_M=range_T2))
    print(f"angle {a:5d} deg  ->  MI = {mi_values[-1]:.4f}")
mi_values = np.array(mi_values)
coarse_best = angles[np.argmax(mi_values)]
print("Coarse best angle:", coarse_best)

# Fine sweep (1° steps) around the coarse optimum
angles_fine = np.arange(coarse_best - 10, coarse_best + 11, 1)
mi_fine = []
for a in angles_fine:
    T2_rot = resample_T2(rotation_matrix(theta_z=a), order=1)
    mi_fine.append(mutual_information(T1_data, T2_rot, bins=32,
                                      range_F=range_T1, range_M=range_T2))
mi_fine = np.array(mi_fine)
best_angle = angles_fine[np.argmax(mi_fine)]
print("Best angle:", best_angle)

True
angle  -180 deg  ->  MI = 0.0992
angle  -175 deg  ->  MI = 0.1028
angle  -170 deg  ->  MI = 0.1048
angle  -165 deg  ->  MI = 0.1063
angle  -160 deg  ->  MI = 0.1054
angle  -155 deg  ->  MI = 0.1032
angle  -150 deg  ->  MI = 0.1028
angle  -145 deg  ->  MI = 0.1056
angle  -140 deg  ->  MI = 0.1089
angle  -135 deg  ->  MI = 0.1100
angle  -130 deg  ->  MI = 0.1082
angle  -125 deg  ->  MI = 0.1030
angle  -120 deg  ->  MI = 0.0983
angle  -115 deg  ->  MI = 0.0982
angle  -110 deg  ->  MI = 0.0993
angle  -105 deg  ->  MI = 0.1005
angle  -100 deg  ->  MI = 0.1016
angle   -95 deg  ->  MI = 0.1022
angle   -90 deg  ->  MI = 0.1017
angle   -85 deg  ->  MI = 0.1000
angle   -80 deg  ->  MI = 0.0976
angle   -75 deg  ->  MI = 0.0954
angle   -70 deg  ->  MI = 0.0948
angle   -65 deg  ->  MI = 0.0940
angle   -60 deg  ->  MI = 0.0927
angle   -55 deg  ->  MI = 0.0917
angle   -50 deg  ->  MI = 0.0918
angle   -45 deg  ->  MI = 0.0913
angle   -40 deg  ->  MI = 0.0922
angle   -35 deg  ->  MI = 0.0940
angle

In [15]:
# Final resampling at the best angle, with cubic interpolation for quality
R_best = rotation_matrix(theta_z=best_angle)
T2_registered = resample_T2(R_best, order=3)

print(f"Best angle: {best_angle} deg")
print(f"MI before: {mutual_information(T1_data, T2_data_resampled, range_F=range_T1, range_M=range_T2):.4f}")
print(f"MI after:  {mutual_information(T1_data, T2_registered, range_F=range_T1, range_M=range_T2):.4f}")

# Same overlay as in Task 1, now with the registered T2
Viewer(T2_registered / T2_registered.max() + T1_data / T1_data.max())

Best angle: 45 deg
MI before: 0.1160
MI after:  0.3505


### Task 5: Perform automatic registration

Plot the negative mutual information, i.e. the energy function $E(\mathbf{w})= H_{F,M}-H_{F}-H_{M}$ for every angle in your grid search space. Select the one with the lowest energy (i.e., the best angle for registration) and transform the T2-weighted volume according to this angle.

Use the Viewer() function to visualize your registration result.

### Task 6: Compare the joint histogram before and after registration

Visualize the joint histogram after registration. Compare the result to your initial joint histogram (i.e., before registration) and discuss your findings in your report.

### Additional task for the enthusiastic student: Compare the MI obtained with landmark-based and intensity-based registration

Perform landmark-based registration between the T1-weighted and T2-weighted volumes instead, just like you did in the last exercise. For the resulting registration, view your results with the Viewer() class; visualize the joint histogram, and calculate the mutual information. 

Compare the results with those obtained using mutual information, and discuss.

> ***Hint:***
> - You can reuse the landmarks that you collected in the landmark-based registration exercise, as these were defined in voxel space. 